# 🚀 Deliverable 3 · Evaluación Oficial en GPU T4 (Qwen3-4B)

Este cuaderno reproduce las evaluaciones oficiales del sistema Multi-Agente en **Google Colab** utilizando la GPU Tesla T4 y el modelo oficial fijado por la asignatura (**Qwen/Qwen3-4B** cuantizado en 4 bits NF4).

### Objetivos de esta corrida:
1. **Parte 1 (P41 a P50):** Verificar empíricamente en GPU la resolución del *livelock* en las 10 preguntas con premisas falsas (salto de 0/10 a 9/10).
2. **Parte 2 (H01 a H25):** Evaluar las 25 consultas difíciles con el validador normalizado y las reglas de parada en modo `agentes_completo`.

Ambas pruebas corren exclusivamente en modo Multi-Agente (`agentes_completo`), completándose en ~15 minutos.

In [ ]:
# 1. Verificación de Acelerador GPU CUDA
import torch
print("PyTorch versión:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ Debes activar GPU T4 en: Entorno de ejecución > Cambiar tipo de entorno > T4 GPU.")
print("GPU detectada:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Instalación de dependencias fijadas por contrato
!pip install --quiet \
    transformers==4.57.6 \
    accelerate==1.11.0 \
    bitsandbytes==0.48.1 \
    sentence-transformers==5.1.2 \
    tokenizers==0.22.2
print("✅ Dependencias fijadas instaladas correctamente.")

In [ ]:
# 3. Desempaquetado del entorno, prompts y conjuntos de preguntas
BUNDLE_B64 = (
'...'
'DSfCpP/j/wPw3oMwLQwRAA=='
)
EXPECTED = '6fba3dc40ce734971837680a1bfe5a4bb1592b88619f1e54cd600293ebd13aee'
FILE_HASHES = {'d3_agentes/__init__.py': 'ee0695533a98ad593db63822f7d3775fe565e1c4a5575f78de54034987d44d9d', 'd3_agentes/bridge.py': '0e15d6153defdb4bc77c0c2a99d32e28e34a31b5f24c97b94fd0e06c23924ca1', 'd3_agentes/common.py': '76f9cb1d436e46f095d37a70ddd9b1ab8c8da0f8e1ce5ee1f8f2f1831077bb20', 'd3_agentes/core.py': 'aee736fd888ff6c73b97193eb4350795343d8ab661cd9d61ac99b0c0f195ccc3', 'd3_agentes/downloads.py': '9345bffa531e1fb55dcd6d572f8afbc9c324972e29c88103aa297486042924cc', 'd3_agentes/experiment.py': 'd3c6b4b1ed314258e175df95193411782ad63ff39d077fb5b5e723a65c4cc8c8', 'd3_agentes/runtime.py': '112401003b7051b2f462f0496bfa1f6a313a7fa01c7c7a262930837fbc04ab6e', 'd3_agentes/tools.py': 'f075926974535b9b03aa8eaf431b0a803af13d81444b08f136818643a09ddab0', 'prompts/auditor.txt': 'b71aed834269f72d691c13b81b375bf1c64b0356749547f491daf6efb81058db', 'prompts/completitud.txt': '72918aa2f8a5bfb7981073823e44145c2574691812672487df6664b540e9c538', 'prompts/descomposicion.txt': 'cf09766bec652248002dfa204b2cf8fff4df6330f22f01c838b1e1c2d4de50f8', 'prompts/investigador.txt': '0bcc8694aa10ecdcfa5c17438071a7390cd77a2a355141191e13b5704ee1c9ea', 'prompts/redactor.txt': '25df34e7d299ddaa25db0f2877dba7965b5129b48ca64e6d92cf7dddb546ef33', 'prompts/verificacion.txt': '8aa50bb8a3fbe4949331186e65ae4a09711be095a04af3830035cec74b1a1587', 'd2_runtime/experimento.py': 'd20ce66a54c7791a480e0d9466e1740b34a8c5198df94ec52c1d4868219e22d0', 'd2_runtime/sistema.py': 'be9ff8714f6580b30618ef4d45c98e744463da620e98721701dadfe9521fe130', 'd2_runtime/recuperador_portable.py': 'a04c720b0cf25eb8010996dd28f70f8a08806b6c556e0be6bc2ea3a6fb4dd056', 'data/corpus.json': '0a31ff812a1969ff81bbc895feaba490e3bd027b092e52101296e3c872719400', 'data/fragmentos_busqueda.json': 'c125615c1c3cddb23d69c3b2b93d7a73e076c90fe861cc217b59f293aa9b14cd', 'data/preguntas.json': '89084eb950e1c1bca00c0a90adbf737a500a978cf12fe61bf70f2325ccf15d12', 'data/configuracion.json': 'd5dd7936c24d811b302cc961b926735a8626708a94fc65b21424dad03eaab310', 'data/preguntas_p41_p50.json': '18c8974fd228fd1f0a2d5db599f7cc7a2c0d0bb307d31647a8e33cc123898a8e', 'data/preguntas_dificiles.json': '70f89171c04b52a70980ac3fc318b0e27973ea8421c7425ab7da7160f4a2f154'}

import base64, gzip, hashlib, json, sys
from pathlib import Path

packed = base64.b64decode(BUNDLE_B64, validate=True)
if hashlib.sha256(packed).hexdigest() != EXPECTED:
    raise ValueError("Paquete alterado o corrupto.")
sources = json.loads(gzip.decompress(packed))
if set(sources) != set(FILE_HASHES):
    raise ValueError("Inventario del paquete alterado.")

WORKDIR = Path.cwd() / f"d3_gpu_runtime_{EXPECTED[:16]}"
for name, text in sources.items():
    path = WORKDIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

sys.path.insert(0, str(WORKDIR.resolve()))
print("✅ Fuentes restauradas en:", WORKDIR.name)

In [ ]:
# 4. Carga de E5 y del modelo oficial Qwen3-4B en GPU T4
from pathlib import Path
from d3_agentes.runtime import load, execute

OUTPUT = Path("/content/d3_evaluacion_runs")
OUTPUT.mkdir(parents=True, exist_ok=True)

ENV = load(OUTPUT)
print("✅ Modelo oficial Qwen3-4B cargado exitosamente en GPU.")

In [ ]:
# 5. PARTE 1: Ejecutar las 10 preguntas con premisas falsas (P41 a P50)
import json, pandas as pd
from d3_agentes.common import VARIANTS

preguntas_p41_p50 = json.loads((WORKDIR / "data/preguntas_p41_p50.json").read_text(encoding="utf-8"))
print(f"🚀 Iniciando evaluación de {len(preguntas_p41_p50)} preguntas con premisas falsas (P41-P50)...")

out_dir_p41_p50 = execute(ENV, preguntas_p41_p50, OUTPUT, variants=("agentes_completo",), tag="eval_p41_p50_gpu", verbose=True)
print("✅ Parte 1 completada en:", out_dir_p41_p50.name)

# Mostrar resumen inmediato
csv_p41 = out_dir_p41_p50 / "agentes_completo.csv"
if csv_p41.exists():
    df_res = pd.read_csv(csv_p41)
    print("\n=== RESULTADOS PREMISAS FALSAS (P41-P50) ===")
    for _, r in df_res.iterrows():
        print(f"P{int(r['pregunta_id'])}: estado={r['estado']} | llamadas={r['llamadas_modelo']} | resp={str(r['respuesta_modelo'])[:60]}...")

In [ ]:
# 6. PARTE 2: Ejecutar las 25 consultas difíciles (H01 a H25)
preguntas_dificiles = json.loads((WORKDIR / "data/preguntas_dificiles.json").read_text(encoding="utf-8"))
print(f"🚀 Iniciando evaluación de {len(preguntas_dificiles)} preguntas difíciles (H01-H25)...")

out_dir_dificiles = execute(ENV, preguntas_dificiles, OUTPUT, variants=("agentes_completo",), tag="eval_dificiles_gpu", verbose=True)
print("✅ Parte 2 completada en:", out_dir_dificiles.name)

# Mostrar resumen inmediato
csv_dif = out_dir_dificiles / "agentes_completo.csv"
if csv_dif.exists():
    df_dif = pd.read_csv(csv_dif)
    print("\n=== RESULTADOS PREGUNTAS DIFÍCILES (H01-H25) ===")
    for idx, r in df_dif.iterrows():
        print(f"H{idx+1:02d} (id {int(r['pregunta_id'])}): estado={r['estado']} | llamadas={r['llamadas_modelo']} | resp={str(r['respuesta_modelo'])[:60]}...")

In [ ]:
# 7. Descargar respaldos ZIP auditables
from google.colab import files
from d3_agentes.downloads import prepare_download

zip_p41 = prepare_download(out_dir_p41_p50, None)
zip_dif = prepare_download(out_dir_dificiles, None)

print("ZIP Premisas Falsas:", zip_p41)
print("ZIP Difíciles:", zip_dif)

try:
    if zip_p41 and zip_p41.exists(): files.download(str(zip_p41))
    if zip_dif and zip_dif.exists(): files.download(str(zip_dif))
except Exception as e:
    print("Descarga manual desde Archivos:", e)